In [ ]:
import os

# Create screenshots folder if it doesn't exist
screenshots_path = '../dashboard\screenshots'
os.makedirs(screenshots_path, exist_ok=True)
print(f"Folder ready: {screenshots_path}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load processed dataset
df = pd.read_csv('../data/processed/zomato_with_refunds.csv')

print(f"Shape: {df.shape}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")

In [ ]:
# Fix 1: Convert Order_Date to datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'], dayfirst=True)

# Fix 2: Convert Age to integer (after handling NaN)
df['Delivery_person_Age'] = df['Delivery_person_Age'].fillna(
    df['Delivery_person_Age'].median()
).astype(int)

# Fix 3: Convert multiple_deliveries to integer
df['multiple_deliveries'] = df['multiple_deliveries'].fillna(0).astype(int)

# Verify fixes
print(df[['Order_Date', 'Delivery_person_Age', 'multiple_deliveries']].dtypes)
print(f"\nSample dates: {df['Order_Date'].head(3).tolist()}")
print(f"Sample ages: {df['Delivery_person_Age'].head(3).tolist()}")

In [ ]:
# Overall refund rate
total_orders = len(df)
total_refunds = df['Refund_Requested'].sum()
refund_rate = df['Refund_Requested'].mean() * 100

print(f"Total Orders: {total_orders:,}")
print(f"Total Refund Requests: {total_refunds:,}")
print(f"Overall Refund Rate: {refund_rate:.2f}%")
print(f"Total Refund Amount: ₹{df['Refund_Amount'].sum():,.2f}")

# Refund breakdown by reason
print("\nRefund Requests by Reason:")
print(df['Refund_Reason'].value_counts())

# Refund rate by city
print("\nRefund Rate by City:")
city_refund = df.groupby('City')['Refund_Requested'].agg(['sum','mean','count'])
city_refund.columns = ['Total_Refunds', 'Refund_Rate', 'Total_Orders']
city_refund['Refund_Rate'] = (city_refund['Refund_Rate'] * 100).round(2)
city_refund = city_refund.sort_values('Refund_Rate', ascending=False)
print(city_refund)

In [ ]:
plt.figure(figsize=(10, 6))

# Count refund reasons
reason_counts = df['Refund_Reason'].value_counts()

# Create bar chart
sns.barplot(x=reason_counts.values, 
            y=reason_counts.index, 
            palette='Reds_r')

# Professional formatting
plt.title('Refund Requests by Reason\n(Higher "Item Not Received" may indicate fraud)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Number of Refund Requests', fontsize=12)
plt.ylabel('Refund Reason', fontsize=12)
plt.tight_layout()

# Save for dashboard screenshots folder
plt.savefig('../dashboard\screenshots\refund_by_reason.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved successfully!")

In [ ]:
# Calculate per-customer refund statistics
customer_stats = df.groupby('Customer_ID').agg(
    Total_Orders=('ID', 'count'),
    Total_Refunds=('Refund_Requested', 'sum'),
    Total_Refund_Amount=('Refund_Amount', 'sum')
).reset_index()

# Calculate refund rate
customer_stats['Refund_Rate'] = (
    customer_stats['Total_Refunds'] / customer_stats['Total_Orders'] * 100
).round(2)

# Apply risk tier from Milestone 3
def assign_risk(rate):
    if rate > 30:
        return 'High Risk'
    elif rate > 10:
        return 'Medium Risk'
    else:
        return 'Low Risk'

customer_stats['Risk_Tier'] = customer_stats['Refund_Rate'].apply(assign_risk)

# Summary
print(customer_stats['Risk_Tier'].value_counts())
print(f"\nTop 10 Highest Refund Rate Customers:")
print(customer_stats.sort_values('Refund_Rate', ascending=False).head(10))

In [ ]:
# Apply minimum order threshold
MIN_ORDERS = 5

filtered_stats = customer_stats[
    customer_stats['Total_Orders'] >= MIN_ORDERS
].copy()

print(f"Customers with {MIN_ORDERS}+ orders: {len(filtered_stats)}")
print(f"\nRisk Tier Breakdown (filtered):")
print(filtered_stats['Risk_Tier'].value_counts())

print(f"\nHigh Risk Customers:")
high_risk = filtered_stats[
    filtered_stats['Risk_Tier'] == 'High Risk'
].sort_values('Refund_Rate', ascending=False)

print(f"Total High Risk Accounts: {len(high_risk)}")
print(high_risk.head(10))

In [ ]:
# Business impact of high risk accounts
high_risk_amount = high_risk['Total_Refund_Amount'].sum()
total_refund_amount = df['Refund_Amount'].sum()
fraud_percentage = (high_risk_amount / total_refund_amount * 100)

print(f"Total Refund Amount (all customers): ₹{total_refund_amount:,.2f}")
print(f"High Risk Accounts Refund Amount: ₹{high_risk_amount:,.2f}")
print(f"Fraud Risk Percentage of Total Refunds: {fraud_percentage:.2f}%")
print(f"Number of Fraud Suspect Accounts: {len(high_risk)}")
print(f"\nIf we recover 50% of high-risk refunds:")
print(f"Potential Recovery: ₹{high_risk_amount * 0.50:,.2f}")